In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import tempfile, sys, shutil, subprocess, hashlib, urllib.request
root = Path('/content/drive/MyDrive/CameraTrap_Phase2')
results = root/'phase2_results'
src = results/'final_sprint_v1/code'
parent = results/'species_head_trials/20261120T211545Z'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(src/'requirements.txt')],check=True)
out = results/'guided_crop_trials'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
local = out/'runtime'
local.mkdir(parents=True)
revision='1b708b5ae1bb96704459a39c6f6e363b62fa379c'
for name in ('fast_candidate/candidate.py','fast_candidate/crop_sam.py','mask_refinement/combined.py',
             'mask_refinement/bf16_encoder.py','crop_student/crop_model.py','crop_student/guided.py',
             'crop_student/deployment.py','crop_student/best.pt'):
    with urllib.request.urlopen(f'https://raw.githubusercontent.com/salehdev3/Capstone_5/{revision}/experiments/{name}',timeout=90) as f:
        (local/Path(name).name).write_bytes(f.read())
sys.path[:0]=[str(local),str(src)]
# Reload candidate modules when a previous experiment ran in this runtime.
for mod in ('candidate','crop_sam','combined','bf16_encoder','crop_model','guided','deployment'):
    sys.modules.pop(mod,None)
from deployment import GuidedPipeline
from common import read,save,sha,context,csv_save
from runtime import DEMO_IDS
from evaluate import mask_evaluate,benchmark,reference
from PIL import Image
policy=read(parent/'policy.json')
baseline=read(parent/'decision.json')
assert sha(parent/'policy.json')==baseline['policy_sha256']=='136e8ba2f579b0a0ea1db1f73cc939057e74f54bafa7015862ecd02eff3be45f'
weight=local/'best.pt'
assert sha(weight)=='18ff2c7c2e9d68146a45a1796f96897c8139edab22d57c18717ed26cd81b8b54'
policy.update(crop_student=dict(path=str(weight),sha256=sha(weight)), runtime_revision=revision,
              parent_policy_sha256=baseline['policy_sha256'])
_,_,rows,_=context(root)
by_id={r['id']:r for r in rows}
cache=Path(tempfile.mkdtemp())
(cache/'images').mkdir()
def stage(ids):
    for ident in set(ids):
        name=ident+'.jpg'
        target=cache/'images'/name
        shutil.copyfile(root/'images'/name,target)
        with Image.open(target) as img:
            assert hashlib.sha256(img.convert('RGB').tobytes()).hexdigest()==by_id[ident]['pixel_sha256']
labels=reference(root,rows,'val_select')
stage(DEMO_IDS+[p.stem for p in labels])
for size in (256,224):
    dest=out/str(size)
    policy['mask']['sam']['size']=size
    save(dest/'policy.json',policy)
    pipe=GuidedPipeline(root,cache,dest/'policy.json')
    speed=benchmark(pipe,rows,cache,dest/'validation_cpu')
    if speed['p95']>2:
        print('Size',size,'CPU p95',speed['p95'],flush=True)
        continue
    masks=mask_evaluate(pipe,labels,by_id,cache,dest/'validation_masks')
    if masks['mean_iou']<=0.7357426095449149:
        continue
    save(out/'freeze.json',dict(policy_sha256=sha(dest/'policy.json'),size=size))
    speed=benchmark(pipe,rows,cache,dest/'final_cpu')
    labels=reference(root,rows,'test')
    stage([p.stem for p in labels])
    masks=mask_evaluate(pipe,labels,by_id,cache,dest/'historical_masks')
    values={'mask_iou_min':masks['mean_iou'],'p95_cpu_seconds_max':speed['p95']}
    gates=[]
    for item in baseline['acceptance']:
        k=item['metric']
        v=values.get(k,item['value'])
        gates.append(dict(metric=k,value=v,target=item['target'],met=v>=item['target'] if k.endswith('_min') else v<=item['target'],
                          source='measured' if k in values else 'inherited from identical upstream policy'))
    csv_save(out/'acceptance.csv',gates)
    verdict=dict(decision='GO' if all(g['met'] for g in gates) else 'NO-GO',independent_test=False,
                 policy_sha256=sha(dest/'policy.json'),parent_policy_sha256=baseline['policy_sha256'],acceptance=gates)
    save(out/'decision.json',verdict)
    print(verdict)
    break
else:
    save(out/'status.json',dict(status='no candidate passed validation speed and quality'))
print('Saved:',out)
